In [2]:
import numpy as np
import time
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

In [4]:
print("Downloading fashion MNIST dataset(this may take 30 to 60 sec)")
fashion_mnist= fetch_openml('Fashion-MNIST', version=1,as_frame=False)
x=fashion_mnist.data
y=fashion_mnist.target.astype(int)
print(f"total dataset size: {x.shape[0]} images , each with{x.shape[1]} pixels.")

total dataset size: 70000 images , each with784 pixels.


In [5]:
x_subset,_,y_subset,_=train_test_split(
    x,y,
    train_size=12000,
    stratify=y,
    random_state=42
)

In [12]:
x_train,x_test,y_train,y_test=train_test_split(
    x_subset,y_subset,
    test_size=2000,
    stratify=y_subset,
    random_state=42
)
print(f"training images: {x_train.shape[0]}")
print(f"testing images: {x_test.shape[0]}")

training images: 10000
testing images: 2000


In [9]:
print(f"before scaling-> Min: {x_train.min()}, Max: {x_train.max()}")
x_train=x_train/255.0
x_test=x_test/255.0
print(f"after scaling-> Min: {x_train.min()}, Max: {x_train.max()}")


before scaling-> Min: 0, Max: 255
after scaling-> Min: 0.0, Max: 1.0


In [20]:
k_values=[1,3,5,7,9,15]
results={}
print(f"{'K value':<8}|{'Accuracy':<10}|{'prediction time(seconds)':<25}")
print("-"*50)
for k in k_values:
    knn=KNeighborsClassifier(n_neighbors=k,metric='euclidean',n_jobs=-1)
    knn.fit(x_train,y_train)
    start_time=time.time()
    y_pred=knn.predict(x_test)
    elapsed_time=time.time()-start_time
    acc=accuracy_score(y_test,y_pred)
    results[k]={
    "accuracy":acc,
    "time":elapsed_time,
    "predictions":y_pred
    }
    print(f"{k:<8} | {acc*100:<9.2f}% | {elapsed_time:<25.2f}")

K value |Accuracy  |prediction time(seconds) 
--------------------------------------------------
1        | 79.30    % | 0.57                     
3        | 80.90    % | 0.60                     
5        | 81.00    % | 0.63                     
7        | 81.10    % | 0.74                     
9        | 80.95    % | 0.64                     
15       | 80.25    % | 0.71                     


In [ ]:
class_names=[
    "tshirts/top","trouser","pullover","dress","coat","sandals","shirt","sneakers","bag","ankle boot"
]
best_k=max(results,key=lamda k:results[k]["accuracy"])
print(f"best k is: {best_k} with {results[best_k]['accuracy']*100:.2f}%accuracy\n")
print("per-class classification report:")
print(classification_report(y_test,results[best_k]["predictions"],
    